# 03. Batch·Iteration·Epoch
직접 작성한 `03_batches.py`의 풀이를 기반으로 데이터가 배치로 나뉘는 과정과 실행 결과를 기록합니다.



## 0. 작업 전 준비사항
이 실습은 작은 데이터를 사용하므로 CPU로 충분합니다. 이번 배치 실습은 텐서를 CPU에 두며, CUDA를 사용할 수 있다는 사실만으로 GPU에서 실행되지는 않습니다.

GPU 환경을 확인하고 싶다면 다음 항목을 점검합니다.
- [ ] GPU 모델 및 드라이버 버전 확인 (`nvidia-smi`, NVIDIA GPU 환경)
- [ ] PyTorch 버전 확인 (`torch.__version__`)
- [ ] 설치된 PyTorch의 CUDA 빌드 버전 확인 (`torch.version.cuda`, CPU 빌드는 `None`)
- [ ] PyTorch의 CUDA 사용 가능 여부 확인 (`torch.cuda.is_available()`)

이번 실습에서는 CPU 설정을 그대로 사용합니다. GPU를 사용할 수 있는 환경인지 확인하는 절차와 실제 실습의 실행 장치는 별개입니다.

### 노트북 실행 순서
PyTorch가 설치된 Python 커널을 선택하고 **Run All(모두 실행)**로 위에서 아래로 실행합니다. 커널을 재시작하면 import와 변수가 초기화되므로 첫 코드 셀부터 다시 실행합니다. 이번 실습은 CPU로 충분하며 CSV 파일이나 `nn` import는 필요하지 않습니다.


### GPU 환경 확인 코드
아래 셀은 NVIDIA GPU 모델·드라이버, PyTorch 버전, CUDA 빌드 버전과 사용 가능 여부를 확인합니다. `nvidia-smi`가 없어도 CPU 실습을 계속할 수 있습니다. CUDA를 사용할 수 있으면 작은 텐서 연산도 실행해 봅니다.


In [1]:
import shutil
import subprocess
import torch

if shutil.which("nvidia-smi"):
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"],
        capture_output=True, text=True,
    )
    print("GPU 모델 / 드라이버:")
    print((result.stdout if result.returncode == 0 else result.stderr).strip()
          or "GPU 정보가 반환되지 않았습니다.")
else:
    print("nvidia-smi 없음: NVIDIA GPU 모델·드라이버 확인 생략")

print("PyTorch 버전:", torch.__version__)
print("PyTorch CUDA 빌드 버전:", torch.version.cuda)
print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU 이름:", torch.cuda.get_device_name(0))
    probe = torch.ones(2, device="cuda")
    print("GPU 연산 확인:", (probe + probe).cpu().tolist())
else:
    print("이번 실습은 CPU에서 실행할 수 있습니다.")


GPU 모델 / 드라이버:
GPU 정보가 반환되지 않았습니다.
PyTorch 버전: 2.14.0+cu130
PyTorch CUDA 빌드 버전: 13.0
CUDA 사용 가능: False
이번 실습은 CPU에서 실행할 수 있습니다.


## 1. 핵심 개념
| 용어 | 의미 | 이번 실습 |
| --- | --- | --- |
| Sample | 데이터 한 건 | 입력 특성 2개와 정답 1개 |
| Batch | 한 번에 처리하는 샘플 묶음 | 최대 6건 |
| Iteration | 배치 하나를 처리하는 반복 한 번 | 한 epoch에 4번 |
| Epoch | 준비한 데이터 전체를 한 번 순회 | 20건을 6·6·6·2건으로 순회 |

앞선 01·02번은 전체 학습 데이터를 한 번에 처리했습니다. 이번에는 DataLoader가 데이터를 여러 배치로 나누는 모습을 확인합니다. **모델이나 Optimizer를 만들지 않으므로 실제 학습과 파라미터 갱신은 수행하지 않습니다.**

## 2. 입력과 정답을 Dataset으로 묶기
- `torch.arange(40, dtype=torch.float32)`는 0부터 39까지의 숫자 40개를 만듭니다.
- `.reshape(20, 2)`는 숫자들을 **20개 샘플 × 2개 특성**으로 배치합니다. 첫 입력은 `[0, 1]`, 마지막 입력은 `[38, 39]`입니다.
- `torch.arange(20) % 2`는 0과 1이 번갈아 나오는 정답 20개를 만듭니다. 정답 텐서의 shape은 `(20,)`입니다.
- `TensorDataset(X, y)`는 같은 위치의 입력과 정답을 한 쌍으로 묶습니다. 두 텐서의 첫 번째 차원인 샘플 수가 같아야 합니다.

이 숫자들은 배치의 개수와 모양을 관찰하기 위한 예제이며 `points.csv`와 별개입니다.


In [2]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# 학습용 공통 데이터와 별개인 '배치 개수만 확인하는' 20개 숫자입니다.
X = torch.arange(40, dtype=torch.float32).reshape(20, 2)
y = torch.arange(20) % 2
dataset = TensorDataset(X, y)


## 3. 직접 작성한 풀이: DataLoader 생성
```python
loader = DataLoader(dataset, batch_size=6, shuffle=False)
```

- `dataset`: 순회할 데이터입니다.
- `batch_size=6`: 배치마다 최대 6개 샘플을 가져옵니다.
- `shuffle=False`: Dataset의 순서를 그대로 유지합니다. 입력과 정답의 짝도 유지됩니다.
- `drop_last`를 지정하지 않았으므로 기본값 `False`가 적용되어 남은 2개도 마지막 배치로 반환합니다.

`enumerate(loader, start=1)`은 배치에 1부터 번호를 붙입니다. `(xb, yb)`에는 해당 배치의 입력과 정답이 들어오며, `tuple(...shape)`은 텐서의 모양을 튜플로 출력합니다.


In [3]:
# DataLoader(dataset, batch_size=6, shuffle=False)를 만드세요.
# 20개를 순서대로 나누면 6·6·6·2개입니다. dataset은 필수, batch_size와 shuffle은 선택 인자입니다.
# >>> B04
# TODO B04: 위 요구사항에 맞게 이 블록만 완성합니다.
loader = DataLoader(dataset, batch_size=6, shuffle=False)
# <<< B04
for iteration, (xb, yb) in enumerate(loader, start=1):
    print("iteration:", iteration, "X:", tuple(xb.shape), "y:", tuple(yb.shape))
print("한 epoch의 iteration:", len(loader))


iteration: 1 X: (6, 2) y: (6,)
iteration: 2 X: (6, 2) y: (6,)
iteration: 3 X: (6, 2) y: (6,)
iteration: 4 X: (2, 2) y: (2,)
한 epoch의 iteration: 4


## 4. 실행 결과 해석
| Iteration | 샘플 위치 (0부터 시작) | 입력 shape | 정답 shape |
| --- | --- | --- | --- |
| 1 | 0~5 | `(6, 2)` | `(6,)` |
| 2 | 6~11 | `(6, 2)` | `(6,)` |
| 3 | 12~17 | `(6, 2)` | `(6,)` |
| 4 | 18~19 | `(2, 2)` | `(2,)` |

총 샘플 수는 `6 + 6 + 6 + 2 = 20`입니다. 마지막 배치는 입력 특성이 줄어든 것이 아니라 **샘플 수가 2개로 줄어든 것**입니다. 입력의 두 번째 차원인 특성 수는 계속 2입니다.

이번처럼 일반적인 Dataset을 고정된 batch size로 묶으면 `drop_last=False`에서 한 epoch의 iteration 수는 **샘플 수 ÷ batch size를 올림한 값**입니다. 따라서 `20 / 6`을 올림하면 4이며 `len(loader)`도 4입니다. `len(dataset)`은 배치 개수가 아닌 샘플 수 20입니다.

### 완료 질문에 대한 설명
**20건, batch_size=6일 때 마지막 배치와 한 epoch의 갱신 횟수는?**

마지막 배치는 2건이고 한 epoch는 4 iteration입니다. 일반적인 학습 루프에서 배치마다 `optimizer.step()`을 한 번 호출한다면 4번 갱신합니다. 다만 이번 코드는 배치 모양만 출력하므로 실제 갱신 횟수는 0입니다. 기울기 누적 등을 사용하면 iteration 수와 갱신 횟수가 달라질 수 있습니다.

## 5. 추가로 확인해 볼 내용
원래 설정의 결과를 기록한 뒤 아래 조건을 하나씩 바꿔 다시 실행해 봅니다.

| 변경 조건 | 한 epoch의 iteration 수 | 마지막 배치 크기 | 순회하는 샘플 수 |
| --- | --- | --- | --- |
| 원래 설정: `batch_size=6` | 4 | 2 | 20 |
| `batch_size=5` | 4 | 5 | 20 |
| `batch_size=20` | 1 | 20 | 20 |
| `batch_size=6, drop_last=True` | 3 | 6 | 18 |

`shuffle=True`로 바꾸면 매 순회마다 샘플 순서를 섞습니다. 입력과 정답을 함께 가져오므로 짝은 유지되며, 이번 설정에서 배치 개수와 각 배치의 크기는 바뀌지 않습니다. 순서까지 재현하려면 seed를 지정한 `torch.Generator`를 DataLoader에 전달할 수 있습니다.

학습 비교에서는 같은 epoch 수라도 batch size가 작아지면 보통 갱신 횟수가 늘어납니다. 따라서 batch size의 영향을 해석할 때는 epoch뿐 아니라 iteration과 실제 `optimizer.step()` 횟수도 함께 확인합니다.
